## Imprecision Data

In [1]:
# ── Environment setup ─────────────────────────────────────────────────────
# ProxiPal is the project's custom analysis library (ProxiPal.py in repo root).

import sys
from pathlib import Path

module_dir = Path.cwd().parent
sys.path.append(str(module_dir))

from ProxiPal import *
# ProxiPal also imports pandas, numpy, scipy, matplotlib — these are
# available after this line without separate import statements.
# See: https://github.com/jeffsmith8/NfL-iPLA-paper


OS:				 Windows-11-10.0.26200-SP0
Python version:		 3.12.6 | packaged by conda-forge | (main, Sep 22 2024, 14:01:26) [MSC v.1941 64 bit (AMD64)]
Date and Time:		 2026-05-26 13:08:12
cwd:		 C:\Users\smith.j\OneDrive - wehi.edu.au\Documents\CFHAC_ProxiPal\ProxiPal\user_documents\NfL-PLA Development Paper\NfL-PLA Development Paper Figure Data\GitHub\NfL-iPLA-paper\NfL-iPLA_imprecision
base_path:		 C:\Users\smith.j\OneDrive - wehi.edu.au\Documents\CFHAC_ProxiPal\ProxiPal
python_folder:		 C:\Users\smith.j\OneDrive - wehi.edu.au\Documents\CFHAC_ProxiPal\ProxiPal\python
templates_folder:	 C:\Users\smith.j\OneDrive - wehi.edu.au\Documents\CFHAC_ProxiPal\ProxiPal\templates
data_folder:		 C:\Users\smith.j\OneDrive - wehi.edu.au\Documents\CFHAC_ProxiPal\ProxiPal\data
samples_folder:		 C:\Users\smith.j\OneDrive - wehi.edu.au\Documents\CFHAC_ProxiPal\ProxiPal\samples
quality_folder:		 C:\Users\smith.j\OneDrive - wehi.edu.au\Documents\CFHAC_ProxiPal\ProxiPal\quality
user_downloads:		 C:\Users\smith.

Imprecision data is held in "Paper_imprecision_data.csv"  
Relevant rows have been filtered from the mastertable and all columns included

In [2]:
imprecision_data = pd.read_csv("imprecision_data.csv", low_memory = False)
imprecision_data.shape

(345, 262)

### Fitting  
4PL and 5PL curve fitting in imprecision_data was performed using custom functions defined in ProxiPal.py. Standard Python packages such as scipy are inadequate for this purpose because the NfL-iPLA standard curves are near-linear, causing generic optimisers to return poorly constrained A, B, C, D parameter estimates. The ProxiPal functions address this by initialising parameter bounds and starting estimates from empirical knowledge of typical NfL-iPLA standard curve geometry.
  
Fitting is performed at when a qPCR experiment is first processed for reporting. Users may choose the default Simple Linear Regression with or without LinReg Cq or N0 corrections. Users testing 4PL or 5PL fits for additional combinations of experimental values, i.e. threshold values, background corrections do so by running the Advanced Processing section of ProxiPal's reporting notebook.  
  
See https://github.com/proxipal-pcr/proxipal/blob/main/app/python/ProxiPal_Report_v2_1.ipynb
  
The ProxiPal_Report_v2_1 notebook also points to the relevant functions inside ProxiPal.

In [3]:
def analyze_sample_stats(data, model_type='SLR; log10(x); exc_std0; ct; raw_ng/L', range_round = 0):
    """
    Calculate statistics for each sample_id with values rounded to 2 decimal places,
    including 1, 2, and 3 standard deviation ranges around the mean
    
    Parameters:
    data (pandas.DataFrame): DataFrame with columns 'sample_id' and measurement values
    model_type (str): Name of the column containing measurement values
    
    Returns:
    pandas.DataFrame: Statistics for each sample_id with standard statistics and std ranges
    All numerical values are rounded to 2 decimal places.
    """
    from scipy import stats
    import re
    
    def calculate_stats(group):
        n = len(group)
        mean = group.mean()
        std = group.std()
        # cv = (std / mean) * 100  # Calculate CV before rounding
        cv = (std / abs(mean)) * 100
        
        # Calculate 95% confidence interval
        ci = stats.t.interval(
            confidence=0.95,
            df=n-1,
            loc=mean,
            scale=stats.sem(group)
        )
        
        mean = round(mean, 1)
        std1 = round(std, 1)
        std2 = round(2*std, 1)
        std3 = round(3*std, 1)

        # Calculate std ranges
        std1_lower = mean - std1
        std1_upper = mean + std1
        std2_lower = mean - (std2)
        std2_upper = mean + (std2)
        std3_lower = mean - (std3)
        std3_upper = mean + (std3)
        
        return pd.Series({
            'mean': round(mean, 2),
            'std': round(std, 2),
            'cv_percent': round(cv, 2),
            'ci_lower': round(ci[0], 1),
            'ci_upper': round(ci[1], 1),
            'n_samples': n,
            'std1_range': [round(std1_lower, 1), round(std1_upper, 1)],
            'std2_range': [round(std2_lower, 1), round(std2_upper, 1)],
            'std3_range': [round(std3_lower, 1), round(std3_upper, 1)]
        })
    
    # Group by sample_id and calculate statistics
    results = []
    for sample_id, group in data.groupby('sample_id'):
        stats_series = calculate_stats(group[model_type])
        stats_series['sample_id'] = sample_id
        results.append(stats_series)
    
    # Create final DataFrame
    stats_df = pd.DataFrame(results)
    stats_df = stats_df[['sample_id', 'mean', 'std', 'cv_percent', 
                        'ci_lower', 'ci_upper', 'n_samples',
                        'std1_range', 'std2_range', 'std3_range']]
    
    # Custom sorting function
    def custom_sort_key(sample_id):
        # First check if 'std' is in sample_id
        has_std = 'std' in sample_id.lower()
        
        # Look for numbers in square brackets
        match = re.search(r'\[([\d.]+)\]', sample_id)
        if has_std:
            # If it's a std sample, put it at the end with numeric sorting
            if match:
                return (1, float(match.group(1)), sample_id)
            return (1, float('inf'), sample_id)
        else:
            # For non-std samples
            if match:
                return (0, float(match.group(1)), sample_id)
            return (0, float('inf'), sample_id)
    
    # Sort using the custom function
    stats_df = stats_df.sort_values('sample_id', key=lambda x: x.map(custom_sort_key)).reset_index(drop=True)
    
    return stats_df

### Calculate CVs for standard linear model  


In [4]:
# Define reseults column
SLR_ct = 'SLR; log10(x); exc_std0; ct; raw_ng/L'

stats_results = analyze_sample_stats(imprecision_data, model_type = SLR_ct, range_round = 2)
stats_results

,sample_id,mean,std,cv_percent,ci_lower,ci_upper,n_samples,std1_range,std2_range,std3_range
0,AD8 blank,0.0,0.00,58.04,0.0,0.0,38,"[0.0, 0.0]","[0.0, 0.0]","[0.0, 0.0]"
1,CAL-QC-0.1,0.1,0.01,13.38,0.1,0.1,30,"[0.1, 0.1]","[0.1, 0.1]","[0.1, 0.1]"
2,CAL-QC-3.2,2.6,0.39,14.83,2.5,2.8,30,"[2.2, 3.0]","[1.8, 3.4]","[1.4, 3.8]"
3,CAL-QC-80,67.5,6.90,10.21,65.0,70.1,30,"[60.6, 74.4]","[53.7, 81.3]","[46.8, 88.2]"
4,NFL-LoQ,1.9,0.34,18.40,1.7,2.0,32,"[1.6, 2.2]","[1.2, 2.6]","[0.9, 2.9]"
5,NFL-QC-H,90.5,9.91,10.94,87.4,93.7,41,"[80.6, 100.4]","[70.7, 110.3]","[60.8, 120.2]"
6,NFL-QC-L,6.2,0.81,13.09,6.0,6.5,38,"[5.4, 7.0]","[4.6, 7.8]","[3.8, 8.6]"
7,NFL-QC-M,19.2,2.19,11.38,18.6,19.9,42,"[17.0, 21.4]","[14.8, 23.6]","[12.6, 25.8]"
8,std1[0]_progen#62008,0.0,0.00,51.97,0.0,0.0,8,"[0.0, 0.0]","[0.0, 0.0]","[0.0, 0.0]"
9,std1[0.128]_progen#62008,0.1,0.01,10.38,0.1,0.1,8,"[0.1, 0.1]","[0.1, 0.1]","[0.1, 0.1]"


### Limit of Blank,  Limit of Detection   
LoB = mean + 1.645(SD)  
LoD = LoB + 1.645(SD low concentration sample)

In [5]:
# import pandas as pd
# from typing import List

def lob_lod(imprecision_data: pd.DataFrame, model_list: List[str], decimals: int = 2) -> pd.DataFrame:
    results = {}

    for model_type in model_list:
        blanks_table = imprecision_data[imprecision_data['sample_id'].isin(['AD8 blank', 'std1[0]_progen#62008'])]
        LoB_stdev = blanks_table[model_type].std()
        LoB_mean = blanks_table[model_type].mean()
        LoB = LoB_mean + 1.645 * LoB_stdev

        cal_qc_low = imprecision_data[imprecision_data['sample_id'] == 'CAL-QC-0.1']
        LoD_cal = LoB + 1.645 * cal_qc_low[model_type].std()

        nfl_qc_low = imprecision_data[imprecision_data['sample_id'] == 'NFL-LoQ']
        LoD_nfl = LoB + 1.645 * nfl_qc_low[model_type].std()

        results[model_type] = {
            "Limit of Blank": round(LoB, decimals),
            "Limit of Detection (Recombinant)": round(LoD_cal, decimals),
            "Limit of Detection (Native)": round(LoD_nfl, decimals)
        }

    return pd.DataFrame(results)


In [6]:
limits_table = lob_lod(imprecision_data, model_list=[SLR_ct], decimals=2)
limits_table

,SLR; log10(x); exc_std0; ct; raw_ng/L
Limit of Blank,0.01
Limit of Detection (Recombinant),0.03
Limit of Detection (Native),0.57


### Extract Imprecision Summary Table for figure (Linear Model)

In [7]:
qc_order = ['NFL-QC-H', 'NFL-QC-M', 'NFL-QC-L', 'NFL-LoQ',
            'CAL-QC-80', 'CAL-QC-3.2', 'CAL-QC-0.1']

qc_table = stats_results[stats_results['sample_id'].isin(qc_order)].set_index('sample_id').loc[qc_order].reset_index()
qc_table[['sample_id','mean', 'std', 'cv_percent', 'n_samples']]

,sample_id,mean,std,cv_percent,n_samples
0,NFL-QC-H,90.5,9.91,10.94,41
1,NFL-QC-M,19.2,2.19,11.38,42
2,NFL-QC-L,6.2,0.81,13.09,38
3,NFL-LoQ,1.9,0.34,18.40,32
4,CAL-QC-80,67.5,6.90,10.21,30
5,CAL-QC-3.2,2.6,0.39,14.83,30
6,CAL-QC-0.1,0.1,0.01,13.38,30


### Linear Model Competitiveness

To benchmark the influence of **curve-fitting strategy** and **PCR efficiency correction** on quantification precision, a panel of candidate models was evaluated across all imprecision experiments. Models vary along two independent axes:

1. **Calibration curve fit** — Simple Linear Regression (SLR), 4-Parameter Logistic (4PL), or 5-Parameter Logistic (5PL)
2. **Threshold / efficiency correction** — raw Ct versus RDML log₂N₀ with sample-mean efficiency (no-plateau, statistical efficiency)

All models exclude std0 replicates (`exc_std0`). The x-axis transformation (linear or log₁₀) is noted per model.

#### Model classification

| Status | Model |
|--------|-------|
| **Recommended** | `SLR; log10(x); exc_std0; ct` |
| **Recommended** | `SLR; log10(x); exc_std0; rdml_log2N0 (mean eff) - no plateau - stat efficiency` |
| Developmental | `4PL; linear(x); exc_std0; rdml_log2N0 (mean eff) - no plateau - stat efficiency` |
| Developmental | `4PL; log10(x); exc_std0; rdml_log2N0 (mean eff) - no plateau - stat efficiency` |
| Developmental | `5PL; linear(x); exc_std0; rdml_log2N0 (mean eff) - no plateau - stat efficiency` |
| Developmental | `5PL; log10(x); exc_std0; rdml_log2N0 (mean eff) - no plateau - stat efficiency` |

The cells below compute, for each model:
- **Imprecision CVs** across the QC panel (`NFL-QC-H/M/L`, `NFL-LoQ`, `CAL-QC-80/3.2/0.1`)
- **LoB / LoD** using the EP17-A2 approach (LoB = mean_blank + 1.645·SD_blank; LoD = LoB + 1.645·SD_low)
- **Aggregate calibration fit statistics** (R², R²-adj, RMSE, AICc, Syx, residual skewness/pattern, complexity ratio, SS_res, SS_tot)


#### Imprecision CVs — SLR · log₁₀(x) · Ct (recommended baseline)

The reference recommended model uses simple linear regression on log₁₀-transformed concentrations with raw Ct as the y-variable. This cell was already run above (`SLR_ct`); here the parallel block for the RDML efficiency-corrected variant is introduced for direct comparison.

#### Imprecision CVs — SLR · log₁₀(x) · RDML log₂N₀ (mean efficiency, recommended)

Same linear fit, but substituting RDML-derived log₂N₀ (computed with sample-mean PCR efficiency, no-plateau assumption, statistical efficiency correction) for raw Ct. This correction accounts for inter-run efficiency variation and is expected to reduce CV at low concentrations where efficiency drift has the greatest proportional impact.

In [8]:
# SLR_rdml
SLR_rdml =  'SLR; log10(x); exc_std0; rdml_log2N0 (mean eff) - no plateau - stat efficiency; raw_ng/L'

stats_results = analyze_sample_stats(imprecision_data, model_type = SLR_rdml, range_round = 2)
qc_table = stats_results[stats_results['sample_id'].isin(qc_order)].set_index('sample_id').loc[qc_order].reset_index()
qc_table[['sample_id','mean', 'std', 'cv_percent', 'n_samples']]

,sample_id,mean,std,cv_percent,n_samples
0,NFL-QC-H,90.0,10.04,11.16,41
1,NFL-QC-M,19.6,2.28,11.65,42
2,NFL-QC-L,6.3,0.79,12.66,38
3,NFL-LoQ,1.9,0.37,19.95,32
4,CAL-QC-80,67.7,7.52,11.11,30
5,CAL-QC-3.2,2.6,0.37,14.29,30
6,CAL-QC-0.1,0.1,0.01,13.50,30


#### Imprecision CVs — 4PL · linear(x) · Ct (developmental)

Four-parameter logistic fit on untransformed concentration (linear x-axis) with raw Ct. The 4PL adds asymptote parameters (upper and lower plateau) to the standard sigmoid, which can improve fit at the dynamic range extremes but adds 2 free parameters relative to SLR, increasing the risk of overfitting on sparse calibrator designs.

In [9]:
# PL4_ct_linear
PL4_ct_linear =  '4PL; linear(x); exc_std0; ct; raw_ng/L'

stats_results = analyze_sample_stats(imprecision_data, model_type = PL4_ct_linear, range_round = 2)
qc_table = stats_results[stats_results['sample_id'].isin(qc_order)].set_index('sample_id').loc[qc_order].reset_index()
qc_table[['sample_id','mean', 'std', 'cv_percent', 'n_samples']]

,sample_id,mean,std,cv_percent,n_samples
0,NFL-QC-H,91.8,10.53,11.46,41
1,NFL-QC-M,19.7,2.46,12.52,42
2,NFL-QC-L,6.4,0.94,14.62,38
3,NFL-LoQ,1.8,0.37,20.04,32
4,CAL-QC-80,66.3,6.15,9.29,30
5,CAL-QC-3.2,2.7,0.44,16.04,30
6,CAL-QC-0.1,0.1,0.01,14.86,30


#### Imprecision CVs — 4PL · linear(x) · RDML log₂N₀ (developmental)

Four-parameter logistic on linear x, with RDML efficiency-corrected log₂N₀. Tests whether the combination of sigmoidal curve shape and efficiency correction yields additive precision gains over either correction applied alone.

In [10]:
# PL4_rdml_linear
PL4_rdml_linear =  '4PL; linear(x); exc_std0; rdml_log2N0 (mean eff) - no plateau - stat efficiency; raw_ng/L'

stats_results = analyze_sample_stats(imprecision_data, model_type = PL4_rdml_linear, range_round = 2)
qc_table = stats_results[stats_results['sample_id'].isin(qc_order)].set_index('sample_id').loc[qc_order].reset_index()
qc_table[['sample_id','mean', 'std', 'cv_percent', 'n_samples']]

,sample_id,mean,std,cv_percent,n_samples
0,NFL-QC-H,91.4,10.36,11.33,41
1,NFL-QC-M,20.0,2.61,13.02,42
2,NFL-QC-L,6.4,0.88,13.80,38
3,NFL-LoQ,1.8,0.40,22.16,32
4,CAL-QC-80,66.7,6.87,10.29,30
5,CAL-QC-3.2,2.7,0.41,15.29,30
6,CAL-QC-0.1,0.1,0.01,16.26,30


#### Imprecision CVs — 4PL · log₁₀(x) · Ct (developmental)

Four-parameter logistic on log₁₀-transformed concentration with raw Ct. The log₁₀ x-transformation linearises the dynamic range, so the sigmoid plateaus are likely to be artifactual rather than biologically meaningful here; this model is included to confirm whether the added parameters improve or degrade imprecision relative to SLR on the same axis.

In [11]:
# PL4_ct_log
PL4_ct_log =  '4PL; log10(x); exc_std0; ct; raw_ng/L'

stats_results = analyze_sample_stats(imprecision_data, model_type = PL4_ct_log, range_round = 2)
qc_table = stats_results[stats_results['sample_id'].isin(qc_order)].set_index('sample_id').loc[qc_order].reset_index()
qc_table[['sample_id','mean', 'std', 'cv_percent', 'n_samples']]

,sample_id,mean,std,cv_percent,n_samples
0,NFL-QC-H,154.9,109.51,70.69,41
1,NFL-QC-M,33.0,23.73,71.92,42
2,NFL-QC-L,11.2,7.93,71.04,38
3,NFL-LoQ,3.2,2.39,75.20,32
4,CAL-QC-80,132.8,94.69,71.28,30
5,CAL-QC-3.2,5.2,3.45,65.97,30
6,CAL-QC-0.1,0.2,0.14,78.73,30


#### Imprecision CVs — 5PL · linear(x) · Ct (developmental)

Five-parameter logistic (adds an asymmetry parameter *f* to the 4PL) on linear x with raw Ct. The 5PL is the most flexible sigmoidal model evaluated; it is appropriate when the response curve is asymmetric around the inflection point. With 5 free parameters and a typically modest calibrator *n*, the AICc penalty for complexity should be carefully assessed.

In [12]:
# PL5_ct_linear
PL5_ct_linear =  '5PL; linear(x); exc_std0; ct; raw_ng/L'

stats_results = analyze_sample_stats(imprecision_data, model_type = PL5_ct_linear, range_round = 2)
qc_table = stats_results[stats_results['sample_id'].isin(qc_order)].set_index('sample_id').loc[qc_order].reset_index()
qc_table[['sample_id','mean', 'std', 'cv_percent', 'n_samples']]

,sample_id,mean,std,cv_percent,n_samples
0,NFL-QC-H,92.2,10.56,11.45,41
1,NFL-QC-M,19.8,2.61,13.17,42
2,NFL-QC-L,6.5,0.95,14.62,38
3,NFL-LoQ,1.8,0.38,20.70,32
4,CAL-QC-80,65.1,6.36,9.76,30
5,CAL-QC-3.2,2.8,0.42,15.33,30
6,CAL-QC-0.1,0.1,0.01,15.09,30


#### Imprecision CVs — 5PL · linear(x) · RDML log₂N₀ (developmental)

Five-parameter logistic on linear x with RDML efficiency correction. The most parameter-rich model in the panel; expected to offer the best raw calibration fit at the cost of highest complexity ratio and greatest overfitting risk.

In [13]:
# PL5_rdml_linear(x)
PL5_rdml_linear =  '5PL; linear(x); exc_std0; rdml_log2N0 (mean eff) - no plateau - stat efficiency; raw_ng/L'

stats_results = analyze_sample_stats(imprecision_data, model_type = PL5_rdml_linear, range_round = 2)
qc_table = stats_results[stats_results['sample_id'].isin(qc_order)].set_index('sample_id').loc[qc_order].reset_index()
qc_table[['sample_id','mean', 'std', 'cv_percent', 'n_samples']]

,sample_id,mean,std,cv_percent,n_samples
0,NFL-QC-H,93.4,10.59,11.35,41
1,NFL-QC-M,20.5,2.84,13.85,42
2,NFL-QC-L,6.8,1.03,15.20,38
3,NFL-LoQ,2.0,0.55,27.11,32
4,CAL-QC-80,69.0,7.91,11.46,30
5,CAL-QC-3.2,2.8,0.39,14.00,30
6,CAL-QC-0.1,0.1,0.02,22.74,30


#### Imprecision CVs — 5PL · log₁₀(x) · Ct (developmental)

Five-parameter logistic on log₁₀-transformed concentration with raw Ct. As with the 4PL log₁₀ variant, the sigmoid shape on a log-linear axis is difficult to justify mechanistically, but is included to complete the combinatorial evaluation.

In [14]:
# PL5_ct_log
PL5_ct_log =  '5PL; log10(x); exc_std0; ct; raw_ng/L'

stats_results = analyze_sample_stats(imprecision_data, model_type = PL5_ct_log, range_round = 2)
qc_table = stats_results[stats_results['sample_id'].isin(qc_order)].set_index('sample_id').loc[qc_order].reset_index()
qc_table[['sample_id','mean', 'std', 'cv_percent', 'n_samples']]

,sample_id,mean,std,cv_percent,n_samples
0,NFL-QC-H,92.1,10.55,11.45,41
1,NFL-QC-M,19.8,2.62,13.20,42
2,NFL-QC-L,6.5,0.95,14.65,38
3,NFL-LoQ,1.8,0.38,20.75,32
4,CAL-QC-80,65.1,6.34,9.73,30
5,CAL-QC-3.2,2.8,0.42,15.33,30
6,CAL-QC-0.1,0.1,0.01,15.10,30


#### LoB and LoD — all candidate models

Limits of Blank (LoB) and Detection (LoD) are calculated for every candidate model using the EP17-A2 nonparametric approach:

```
LoB = mean(blank) + 1.645 × SD(blank)
LoD = LoB + 1.645 × SD(low-concentration sample)
```

Two low-concentration anchors are used:
- `CAL-QC-0.1` — recombinant NfL calibrator at the lowest concentration
- `NFL-LoQ` — native NfL plasma sample at the limit-of-quantification level

Results are printed per model for direct cross-model comparison.

In [15]:
models_list = ['SLR; log10(x); exc_std0; ct; raw_ng/L',
              'SLR; log10(x); exc_std0; rdml_log2N0 (mean eff) - no plateau - stat efficiency; raw_ng/L',
              '4PL; linear(x); exc_std0; ct; raw_ng/L',
              '5PL; linear(x); exc_std0; ct; raw_ng/L',
              '4PL; linear(x); exc_std0; rdml_log2N0 (mean eff) - no plateau - stat efficiency; raw_ng/L',
              '5PL; linear(x); exc_std0; rdml_log2N0 (mean eff) - no plateau - stat efficiency; raw_ng/L',
              '4PL; log10(x); exc_std0; ct; raw_ng/L',
              '5PL; log10(x); exc_std0; ct; raw_ng/L']


for i in models_list:
    limits_table = lob_lod(imprecision_data, model_list=[i], decimals=3)
    print(i)
    print(limits_table)

SLR; log10(x); exc_std0; ct; raw_ng/L
                                  SLR; log10(x); exc_std0; ct; raw_ng/L
Limit of Blank                                                    0.011
Limit of Detection (Recombinant)                                  0.030
Limit of Detection (Native)                                       0.573
SLR; log10(x); exc_std0; rdml_log2N0 (mean eff) - no plateau - stat efficiency; raw_ng/L
                                  SLR; log10(x); exc_std0; rdml_log2N0 (mean eff) - no plateau - stat efficiency; raw_ng/L
Limit of Blank                                                                0.011                                       
Limit of Detection (Recombinant)                                              0.030                                       
Limit of Detection (Native)                                                   0.622                                       
4PL; linear(x); exc_std0; ct; raw_ng/L
                                  4PL; linear(x); exc_

---
## Calibration Fit Statistics — Cross-Experiment Aggregation
  
Statistical functions for Parameter Logistic fits are described in `ProxiPal.py`, by the functions `calc_PL_metrics()` and
`get_PL_metrics()`
  
The imprecision CV tables above describe **measurement precision** (variability of concentration estimates across replicates). The cells in this section characterise **calibration curve quality** — how well each model fits the standard data used to back-calculate concentrations. These two dimensions are complementary: a model can have an excellent calibration fit but poor imprecision (e.g., if efficiency correction introduces run-specific noise), or vice versa.

### Strategy

For each experiment in `imprecision_data`, fit statistics are embedded in the master table as columns named:
`{model}; {statistic}` (e.g., `SLR; log10(x); exc_std0; ct; R-squared`).

The pipeline below:
1. **Extracts** per-experiment fit statistics for each model into a `model_stats_dict` keyed by model name
2. **Aggregates** across experiments using statistic-appropriate rules (mean for relative metrics; sum for additive quantities; `None` for non-aggregatable fields)
3. **Collates** into `aggregate_summary_df` — a single table with models as columns and fit statistics as rows, enabling direct model comparison

#### Aggregation rules

| Statistic | Aggregation | Rationale |
|-----------|-------------|----------|
| R², R²-adj | mean | Proportion of variance explained — meaningful as an average |
| RMSE | mean | Scale-matched residual error — average reflects typical fit quality |
| AICc | sum | Information-theoretic score accumulates across experiments |
| Standard Error of Estimate (Syx) | mean | Average calibration uncertainty |
| Residual skewness | mean | Average asymmetry of residuals |
| Complexity ratio | mean | Avg parameter-to-residual-df ratio; overfitting risk indicator |
| SS_res, SS_tot | sum | Additive across runs |
| Residual pattern, DoF, equation | — | Non-aggregatable; inspected per-experiment |


In [16]:
models_list = ['SLR; log10(x); exc_std0; ct',
               'SLR; log10(x); exc_std0; rdml_log2N0 (mean eff) - no plateau - stat efficiency',
               '4PL; linear(x); exc_std0; ct',
               '5PL; linear(x); exc_std0; ct',
               '4PL; linear(x); exc_std0; rdml_log2N0 (mean eff) - no plateau - stat efficiency',
               '5PL; linear(x); exc_std0; rdml_log2N0 (mean eff) - no plateau - stat efficiency',
               '4PL; log10(x); exc_std0; ct',
               '5PL; log10(x); exc_std0; ct']

statistic_list = ['R-squared', 'R-squared-adj', 'RMSE', 'AICc',
                  'Standard Error of the Estimate', 'Degrees of Freedom',
                  'residual_pattern', 'residual_skewness', 'complexity_ratio',
                  'Sum of Squares, residual', 'Sum of Squares, total', 'equation']

model_stats_dict = {}

# Group data by experiment to simplify lookup
grouped = imprecision_data.groupby("expt_folder_csv")

for model in models_list:
    # Store per-statistic values for each experiment
    stat_table = pd.DataFrame(index=statistic_list, columns=grouped.groups.keys())

    for expt, group_df in grouped:
        for stat in statistic_list:
            col_name = f"{model}; {stat}"
            if col_name in group_df.columns:
                value = group_df[col_name].iloc[0]  # All rows have the same value, so take the first
                stat_table.loc[stat, expt] = value
            else:
                stat_table.loc[stat, expt] = pd.NA

    model_stats_dict[model] = stat_table

# Example usage:
# model_stats_dict['4PL; linear(x); exc_std0; ct']


In [17]:
model_stats_dict.keys()

dict_keys(['SLR; log10(x); exc_std0; ct', 'SLR; log10(x); exc_std0; rdml_log2N0 (mean eff) - no plateau - stat efficiency', '4PL; linear(x); exc_std0; ct', '5PL; linear(x); exc_std0; ct', '4PL; linear(x); exc_std0; rdml_log2N0 (mean eff) - no plateau - stat efficiency', '5PL; linear(x); exc_std0; rdml_log2N0 (mean eff) - no plateau - stat efficiency', '4PL; log10(x); exc_std0; ct', '5PL; log10(x); exc_std0; ct'])

#### Inspect per-experiment stats for a single model

Verify the extraction worked correctly by inspecting the raw per-experiment table for one model before aggregation. Each column is an experiment run; each row is a fit statistic.

In [18]:
model_stats_dict['5PL; linear(x); exc_std0; rdml_log2N0 (mean eff) - no plateau - stat efficiency']

,241107_e87_kruti,241112_e88_kruti,241114_e89_kruti,241114_e90_kruti
R-squared,0.9999,0.9999,0.9999,0.9999
R-squared-adj,0.9998,0.9997,0.9997,0.9998
RMSE,0.0324,0.0354,0.0365,0.0338
AICc,-20.0102,-18.7774,-18.3398,-19.4045
Standard Error of the Estimate,0.0495,0.0541,0.0558,0.0517
Degrees of Freedom,3.0,3.0,3.0,3.0
residual_pattern,0.0493,0.066,0.0884,0.1776
residual_skewness,0.0325,0.0504,0.0108,0.049
complexity_ratio,0.5714,0.5714,0.5714,0.5714
"Sum of Squares, residual",0.0074,0.0088,0.0093,0.008


#### Aggregate fit statistics across experiments

Apply the aggregation rules defined above to collapse each model's per-experiment fit statistics into a single summary value. The `aggregate` column is appended to each model's stat table in `model_stats_dict`.

In [19]:
aggregation_rules = {
    'R-squared': 'mean',
    'R-squared-adj': 'mean',
    'RMSE': 'mean',
    'AICc': 'sum',
    'Standard Error of the Estimate': 'mean',
    'residual_skewness': 'mean',
    'complexity_ratio': 'mean',
    'Sum of Squares, residual': 'sum',
    'Sum of Squares, total': 'sum',
    'residual_pattern': None,
    'Degrees of Freedom': None,
    'equation': None
}

for key, df in model_stats_dict.items():
    aggregate_values = []

    for stat in df.index:
        rule = aggregation_rules.get(stat)

        if rule == 'mean':
            val = pd.to_numeric(df.loc[stat], errors='coerce').mean()
        elif rule == 'sum':
            val = pd.to_numeric(df.loc[stat], errors='coerce').sum()
        else:
            val = None

        aggregate_values.append(val)

    df['aggregate'] = aggregate_values
    model_stats_dict[key] = df


In [20]:
model_stats_dict['5PL; linear(x); exc_std0; rdml_log2N0 (mean eff) - no plateau - stat efficiency']

,241107_e87_kruti,241112_e88_kruti,241114_e89_kruti,241114_e90_kruti,aggregate
R-squared,0.9999,0.9999,0.9999,0.9999,0.999900
R-squared-adj,0.9998,0.9997,0.9997,0.9998,0.999750
RMSE,0.0324,0.0354,0.0365,0.0338,0.034525
AICc,-20.0102,-18.7774,-18.3398,-19.4045,-76.531900
Standard Error of the Estimate,0.0495,0.0541,0.0558,0.0517,0.052775
Degrees of Freedom,3.0,3.0,3.0,3.0,NaN
residual_pattern,0.0493,0.066,0.0884,0.1776,NaN
residual_skewness,0.0325,0.0504,0.0108,0.049,0.035675
complexity_ratio,0.5714,0.5714,0.5714,0.5714,0.571400
"Sum of Squares, residual",0.0074,0.0088,0.0093,0.008,0.033500


#### Collate into cross-model summary table

`aggregate_summary_df` places all models side-by-side with fit statistics as the index. This is the primary comparison table: lower AICc, higher R²-adj, lower RMSE, near-zero residual skewness, and low complexity ratio all favour a model.

In [21]:
# Create a new DataFrame to hold the aggregates
aggregate_summary_df = pd.DataFrame()

for key, df in model_stats_dict.items():
    # Extract the 'aggregate' column as a Series and name it by the model (key)
    aggregate_summary_df[key] = df['aggregate']

# Display or use aggregate_summary_df
# Example: print(aggregate_summary_df)


In [22]:
aggregate_summary_df

,SLR; log10(x); exc_std0; ct,SLR; log10(x); exc_std0; rdml_log2N0 (mean eff) - no plateau - stat efficiency,4PL; linear(x); exc_std0; ct,5PL; linear(x); exc_std0; ct,4PL; linear(x); exc_std0; rdml_log2N0 (mean eff) - no plateau - stat efficiency,5PL; linear(x); exc_std0; rdml_log2N0 (mean eff) - no plateau - stat efficiency,4PL; log10(x); exc_std0; ct,5PL; log10(x); exc_std0; ct
R-squared,NaN,NaN,0.999850,0.999800,0.999875,0.999900,0.999875,0.999800
R-squared-adj,NaN,NaN,0.999550,0.999325,0.999650,0.999750,0.999575,0.999325
RMSE,NaN,NaN,0.050850,0.063375,0.040100,0.034525,0.050050,0.063375
AICc,0.0,0.0,-65.066400,-50.306300,-70.955000,-76.531900,-66.404200,-50.307800
Standard Error of the Estimate,NaN,NaN,0.077725,0.096800,0.061250,0.052775,0.076500,0.096800
Degrees of Freedom,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
residual_pattern,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
residual_skewness,NaN,NaN,0.055975,0.082675,0.055300,0.035675,0.050725,0.082075
complexity_ratio,NaN,NaN,0.571400,0.571400,0.571400,0.571400,0.571400,0.571400
"Sum of Squares, residual",0.0,0.0,0.098900,0.136000,0.050200,0.033500,0.097400,0.136000


---
## Development: Per-Run Linear Metrics from Raw Fit Parameters

The `model_stats_dict` approach above reads pre-computed statistics embedded in the master table by ProxiPal. The following section provides an **independent recalculation** of linear fit metrics directly from the SLR gradient and intercept stored in the metatable, without relying on pre-embedded columns. This serves as a validation cross-check and as a reusable function for models or experiments not yet processed by the full ProxiPal pipeline.

### `calc_linear_metrics` — standalone metric calculator

Takes raw arrays of observed and predicted values and returns the full suite of fit diagnostics. Inputs required: `y_true`, `y_pred`, `popt` (model parameters), `x` (predictor), `n_params`.

In [23]:
import numpy as np
from scipy.stats import skew
from typing import Dict

def calc_linear_metrics(y_true: np.ndarray, y_pred: np.ndarray, 
                        popt: np.ndarray, x: np.ndarray,
                        n_params: int = 2) -> Dict:
    """
    Calculate fit metrics for a simple linear regression model.

    Parameters:
    - y_true: Observed Ct values
    - y_pred: Predicted Ct values from model
    - popt: Model parameters [slope, intercept]
    - x: Predictor values (e.g., log10(concentration))
    - n_params: Number of parameters (default = 2 for linear)
    """
    n = len(y_true)
    residuals = y_true - y_pred
    ss_res = np.sum(residuals ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)

    r_squared = 1 - (ss_res / ss_tot)
    r_squared_adj = 1 - ((1 - r_squared) * (n - 1) / (n - n_params))
    rmse = np.sqrt(ss_res / n)
    syx = np.sqrt(ss_res / (n - n_params))

    # AIC and AICc
    aic = n * np.log(ss_res / n) + 2 * n_params
    aicc = aic + (2 * n_params * (n_params + 1)) / (n - n_params - 1) if (n - n_params - 1) > 0 else np.nan

    residual_skewness = skew(residuals)
    residual_pattern = np.corrcoef(x, residuals)[0, 1] if n > 1 else np.nan
    complexity_ratio = n_params / (n - n_params) if (n - n_params) > 0 else np.nan

    return {
        'r_squared': r_squared,
        'r_squared_adj': r_squared_adj,
        'rmse': rmse,
        'aicc': aicc,
        'residual_skewness': residual_skewness,
        'residual_pattern': residual_pattern,
        'dof': n - n_params,
        'complexity_ratio': complexity_ratio,
        'syx': syx,
        'ss_res': ss_res,
        'ss_tot': ss_tot,
        'residuals': residuals
    }


#### Test case: experiment e87 (241107_e87_kruti)

Load a single experiment's data and regenerate the SLR metatable to provide input arrays for `calc_linear_metrics_from_metatable` below.

In [24]:
e87_df = imprecision_data[imprecision_data['expt_folder_csv'] == '241107_e87_kruti']

SLR_metatable = calc_metatable_SLR(e87_df, threshold_type = 'ct', std0_status = 'exc_std0',  export = False)


### `calc_linear_metrics_from_metatable` — per-standard metrics from metatable

Iterates over each standard within an experiment's SLR metatable. For each standard, it reads the stored gradient and intercept, reconstructs predicted values, and computes the full metric suite independently. Returns one row per standard.

**Key design decisions:**
- x-axis transformation is inferred from `threshold_type`: `_N0` suffix → `linear(x)`; otherwise → `log10(x)`
- Rows with ambiguous or missing fit parameters are skipped rather than imputed
- AICc falls back to `nan` when `n - n_params - 1 ≤ 0` (too few degrees of freedom for small calibrator sets)

In [25]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from scipy.stats import skew
from typing import Dict

def calc_linear_metrics_from_metatable(SLR_metatable: pd.DataFrame,
                                        threshold_type: str = 'ct',
                                        std0_status: str = 'exc_std0') -> pd.DataFrame:
    """
    Calculate linear regression metrics per standard from an SLR_metatable.
    
    Parameters:
    - SLR_metatable: Output of calc_metatable_SLR()
    - threshold_type: e.g., 'ct' or 'rdml_Cq (mean eff)'
    - std0_status: 'exc_std0' or 'inc_std0'

    Returns:
    - pd.DataFrame: One row per standard with computed regression metrics
    """

    # Determine transformation axis
    transform_x = 'linear(x)' if '_N0' in threshold_type else 'log10(x)'

    # Define relevant columns
    x_col = 'py_known_conc' if '_N0' in threshold_type else 'py_known_conc_log10'
    y_col = threshold_type + '; mean'
    
    grad_col = f"SLR; {transform_x}; {std0_status}; {threshold_type}; grad"
    yint_col = f"SLR; {transform_x}; {std0_status}; {threshold_type}; y_int"
    eqn_col  = f"SLR; {transform_x}; {std0_status}; {threshold_type}; equation"

    # Restrict to usable rows
    usable = SLR_metatable[
        (SLR_metatable['usr_ignore'] != 1) &
        (SLR_metatable[x_col].notna()) &
        (SLR_metatable[y_col].notna())
    ].copy()

    # Group by std
    results = []

    for std in usable['usr_std'].unique():
        subset = usable[usable['usr_std'] == std]

        # Ensure required metadata is available
        if not all(col in subset.columns for col in [grad_col, yint_col]):
            continue

        # Drop rows without gradient/intercept
        m = subset[grad_col].dropna().unique()
        b = subset[yint_col].dropna().unique()

        if len(m) != 1 or len(b) != 1:
            continue  # ambiguous fit — skip

        m = float(m[0])
        b = float(b[0])

        x = subset[x_col].values
        y = subset[y_col].values
        y_pred = m * x + b
        residuals = y - y_pred

        n = len(y)
        n_params = 2
        ss_res = np.sum(residuals ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        r_squared = 1 - ss_res / ss_tot
        r_squared_adj = 1 - ((1 - r_squared) * (n - 1) / (n - n_params))
        rmse = np.sqrt(ss_res / n)
        syx = np.sqrt(ss_res / (n - n_params))
        aic = n * np.log(ss_res / n) + 2 * n_params
        aicc = aic + (2 * n_params * (n_params + 1)) / (n - n_params - 1) if (n - n_params - 1) > 0 else np.nan
        skewness = skew(residuals)
        pattern_corr = np.corrcoef(x, residuals)[0, 1] if n > 1 else np.nan
        complexity_ratio = n_params / (n - n_params) if (n - n_params) > 0 else np.nan

        results.append({
            'usr_std': std,
            'n_points': n,
            'slope': m,
            'intercept': b,
            'r_squared': r_squared,
            'r_squared_adj': r_squared_adj,
            'rmse': rmse,
            'syx': syx,
            'aicc': aicc,
            'residual_skewness': skewness,
            'residual_pattern': pattern_corr,
            'complexity_ratio': complexity_ratio,
            'ss_res': ss_res,
            'ss_tot': ss_tot
        })

    return pd.DataFrame(results)


#### Cross-experiment summary — SLR · Ct

Loop over all experiments, compute per-standard linear metrics, take the first available standard's metrics as the experiment representative, and collate into a wide-form DataFrame (`longform_metrics_df`) with metrics as the row index and experiments as columns.

> **Note:** taking `iloc[0]` selects the first standard by iteration order. If consistent cross-experiment comparison is required, consider explicitly selecting a fixed standard (e.g., `usr_std == 1`) rather than relying on iteration order.

In [26]:
summary_metrics = {}  # Dictionary to hold metrics per expt

for expt in imprecision_data['expt_folder_csv'].unique().tolist():
    expt_df = imprecision_data[imprecision_data['expt_folder_csv'] == expt]
    expt_metrics_df = calc_linear_metrics_from_metatable(expt_df, threshold_type='ct', std0_status='exc_std0')
    
    if not expt_metrics_df.empty:
        # Choose one row per expt — e.g., std1 or the first available
        row = expt_metrics_df.iloc[0]  # or expt_metrics_df[expt_metrics_df['usr_std'] == 1].iloc[0]
        summary_metrics[expt] = row.drop(labels=['usr_std', 'n_points'])  # Drop ID columns

# Create long-form DataFrame: metrics as index, experiments as columns
longform_metrics_df = pd.DataFrame(summary_metrics)
longform_metrics_df

,241107_e87_kruti,241112_e88_kruti,241114_e89_kruti,241114_e90_kruti
slope,-3.360864e+00,-3.423302e+00,-3.374200e+00,-3.436281e+00
intercept,2.498853e+01,2.549171e+01,2.462437e+01,2.477205e+01
r_squared,9.997542e-01,9.993294e-01,9.999012e-01,9.998144e-01
r_squared_adj,9.997338e-01,9.992736e-01,9.998929e-01,9.997989e-01
rmse,7.366263e-02,1.239640e-01,4.689834e-02,6.545095e-02
syx,7.956474e-02,1.338965e-01,5.065599e-02,7.069510e-02
aicc,-6.794036e+01,-5.336648e+01,-8.058274e+01,-7.124981e+01
residual_skewness,-6.583087e-02,1.226867e+00,4.601517e-01,7.099950e-01
residual_pattern,1.774933e-15,-2.066764e-15,-8.096735e-15,-1.367208e-14
complexity_ratio,1.666667e-01,1.666667e-01,1.666667e-01,1.666667e-01


#### Cross-experiment summary — SLR · RDML log₂N₀ (mean efficiency)

Identical pipeline to the Ct version above, substituting RDML efficiency-corrected log₂N₀ as the y-variable. Comparing `longform_metrics_df` from both runs directly quantifies the effect of efficiency correction on calibration fit quality.

In [27]:
summary_metrics = {}  # Dictionary to hold metrics per expt

for expt in imprecision_data['expt_folder_csv'].unique().tolist():
    expt_df = imprecision_data[imprecision_data['expt_folder_csv'] == expt]
    expt_metrics_df = calc_linear_metrics_from_metatable(expt_df, threshold_type='rdml_log2N0 (mean eff) - no plateau - stat efficiency', std0_status='exc_std0')
    
    if not expt_metrics_df.empty:
        # Choose one row per expt — e.g., std1 or the first available
        row = expt_metrics_df.iloc[0]  # or expt_metrics_df[expt_metrics_df['usr_std'] == 1].iloc[0]
        summary_metrics[expt] = row.drop(labels=['usr_std', 'n_points'])  # Drop ID columns

# Create long-form DataFrame: metrics as index, experiments as columns
longform_metrics_df = pd.DataFrame(summary_metrics)
longform_metrics_df

,241107_e87_kruti,241112_e88_kruti,241114_e89_kruti,241114_e90_kruti
slope,-2.762005e+00,-2.759475e+00,-2.831728e+00,-2.924676e+00
intercept,2.351358e+01,2.355055e+01,2.367159e+01,2.410038e+01
r_squared,9.996178e-01,9.995811e-01,9.998373e-01,9.998986e-01
r_squared_adj,9.995860e-01,9.995462e-01,9.998238e-01,9.998901e-01
rmse,7.549792e-02,7.896789e-02,5.049093e-02,4.118203e-02
syx,8.154708e-02,8.529507e-02,5.453644e-02,4.448168e-02
aicc,-6.725130e+01,-6.599308e+01,-7.851601e+01,-8.422218e+01
residual_skewness,5.087669e-01,1.075692e+00,3.132920e-01,5.442968e-01
residual_pattern,-2.009253e-14,8.014177e-15,2.269668e-14,-3.102940e-15
complexity_ratio,1.666667e-01,1.666667e-01,1.666667e-01,1.666667e-01
